# Skyvonyx - train and measure your own detector

```
labelled scenes ──► tile at 1024px / 0.2 overlap ──► split BY SCENE ──► train YOLOv8 (resumable)
   (DOTA or COCO)    same slicer + stretch as the       train / val / test      checkpoints on Drive
                     inference pipeline
                                         ──► measure twice ──► publish to models/best.pt
                                              1. tile-level mAP (model.val)
                                              2. scene-level P/R/F1/AP50 through the WHOLE pipeline
```

Number 2 is the one you may quote to a customer, always with the test set described next to it.
Number 1 helps with training decisions only.

**One sensor, one class, one model to start with.** A ship detector for Sentinel-1 SAR and a building
detector for optical imagery are different models. Train the one your first pilot needs.

**Licences:** check every dataset's terms before training on it (`DATASET_LICENCE` below is written into the
model card). DOTA and xView are non-commercial; weights trained on them carry that restriction.

In [ ]:
#@title 1. Runtime + code (same as the worker notebook, cells 1-2)
import os, subprocess, sys, pathlib, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch to T4")
SKYVONYX_REPO = "https://github.com/harishrajCEO/skyvonyx.git"  #@param {type:"string"}
WORK = pathlib.Path("/content/skyvonyx")
def sh(c): print("$", c); subprocess.run(c, shell=True, check=True)
if not WORK.exists():
    sh(f"git clone --depth 1 {SKYVONYX_REPO} {WORK}")
VENDOR = WORK / "vendor"; VENDOR.mkdir(exist_ok=True)
if not (VENDOR / "sahi").exists():
    sh(f"git clone --depth 1 -q https://github.com/harishrajCEO/sahi.git {VENDOR/'sahi'}")
sh(f"pip install -q -r {WORK}/requirements-colab.txt")
sh(f"pip install -q -e {VENDOR}/sahi")
sys.path.insert(0, str(WORK)); sys.path.insert(0, str(VENDOR / "sahi"))
sh(f"cd {WORK} && python tests/test_training_eval.py | tail -1")

In [ ]:
#@title 2. Drive + what you are training
from google.colab import drive; drive.mount("/content/drive")
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/skyvonyx")

RUN_NAME      = "ship_s1_v1"        #@param {type:"string"}
LABEL_FORMAT  = "dota"              #@param ["dota", "coco"]
IMAGES_DIR    = "/content/drive/MyDrive/datasets/DOTA/train/images"     #@param {type:"string"}
LABELS        = "/content/drive/MyDrive/datasets/DOTA/train/labelTxt"   #@param {type:"string"}
# ^ dota: folder of .txt  |  coco: path to the annotations .json
CLASSES       = "ship"              #@param {type:"string"}
# comma separated; labels with other class names are dropped (and counted in the card)
SENSOR        = "optical"           #@param ["optical", "sar"]
# "sar" = LINEAR backscatter GeoTIFF (e.g. Sentinel-1 GRD) -> dB stretch.
# Already-8-bit SAR chips (HRSID, SSDD jpg/png) are "optical" here: they are rendered already.
# Whatever you pick MUST match the "sensor" you later put in job.json, or train and inference differ.
TILE          = 1024                #@param {type:"integer"}
DATASET_SOURCE  = "DOTA v1.5 train" #@param {type:"string"}
DATASET_LICENCE = "DOTA: academic/non-commercial - verify"  #@param {type:"string"}

classes = [c.strip() for c in CLASSES.split(",") if c.strip()]
RUN_DIR = ROOT / "training_runs" / RUN_NAME; RUN_DIR.mkdir(parents=True, exist_ok=True)
print("classes:", classes, "| sensor:", SENSOR, "| tile:", TILE, "| run dir:", RUN_DIR)

In [ ]:
#@title 3. Build the tiled dataset on local disk (Drive is too slow to train from)
import json, shutil
from skyvonyx import datasets

if LABEL_FORMAT == "dota":
    scenes = datasets.manifest_from_dota(IMAGES_DIR, LABELS)
else:
    scenes = datasets.load_coco(LABELS, IMAGES_DIR)
print(len(scenes), "labelled scenes,", sum(len(s.boxes) for s in scenes), "boxes before class filter")

DS = Path("/content/ds_" + RUN_NAME)
if DS.exists(): shutil.rmtree(DS)
card = datasets.build_dataset(
    scenes, DS, classes,
    spec=datasets.TileSpec(tile=TILE, overlap=0.2, sensor=SENSOR, min_visible=0.5, negative_frac=0.1),
    source=DATASET_SOURCE, licence=DATASET_LICENCE)
shutil.copy(DS / "dataset_card.json", RUN_DIR / "dataset_card.json")
shutil.copy(DS / "test_scenes_gold.json", RUN_DIR / "test_scenes_gold.json")
print(json.dumps(card["splits"], indent=1))
print("dropped (class not in list):", card["dropped_unknown_class"])
if card["splits"].get("test", {}).get("objects", 0) < 100:
    print("!! fewer than 100 test objects - scores will be too noisy to quote. Add labelled scenes.")

In [ ]:
#@title 4. Train (resumes automatically after a disconnect - just re-run cells 1-4)
from ultralytics import YOLO   # AGPL-3.0
BASE_WEIGHTS = "yolov8s.pt"   #@param ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]
EPOCHS = 50                   #@param {type:"integer"}
BATCH  = 8                    #@param {type:"integer"}
# T4 at 1024px: yolov8s batch 8 fits; drop to 4 on an out-of-memory error.

last = RUN_DIR / "train" / "weights" / "last.pt"
if last.exists():
    print("resuming from", last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO(BASE_WEIGHTS).train(
        data=str(DS / "data.yaml"), imgsz=TILE, epochs=EPOCHS, batch=BATCH,
        project=str(RUN_DIR), name="train", exist_ok=True,
        save_period=1,            # a checkpoint every epoch, on Drive
        patience=15, amp=True, workers=2, seed=0, deterministic=True,
        # satellite tiles have no "up": flips both ways are free augmentation
        fliplr=0.5, flipud=0.5, degrees=0.0, mosaic=1.0)
best = RUN_DIR / "train" / "weights" / "best.pt"
print("best:", best, best.exists())

In [ ]:
#@title 5. Measurement 1 - tile-level mAP on the held-out TEST split (for training decisions)
m = YOLO(str(best)).val(data=str(DS / "data.yaml"), split="test", imgsz=TILE, batch=BATCH, plots=False)
tile_metrics = {"mAP50": round(float(m.box.map50), 4), "mAP50_95": round(float(m.box.map), 4),
                "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
                "per_class_ap50": {m.names[int(c)]: round(float(a), 4)
                                   for c, a in zip(m.box.ap_class_index, m.box.ap50)}}
(RUN_DIR / "tile_metrics_test.json").write_text(json.dumps(tile_metrics, indent=2))
print(json.dumps(tile_metrics, indent=2))

In [ ]:
#@title 6. Measurement 2 - the customer number: whole test SCENES through the real pipeline
# slicing + detection + SAHI re-stitch, scored against the full-scene gold. Duplicates the merge
# misses count as false positives here, exactly as they would in a delivery.
import tempfile
from skyvonyx.config import JobConfig
from skyvonyx.pipeline import run_job
from skyvonyx import evaluate

gold_path = RUN_DIR / "test_scenes_gold.json"
gold_coco = json.loads(gold_path.read_text())
job = Path(tempfile.mkdtemp()) / "eval_job"; (job / "input").mkdir(parents=True)
for im in gold_coco["images"]:
    shutil.copy(im["path"], job / "input" / Path(im["path"]).name)

CONF = 0.25   #@param {type:"number"}
cfg = JobConfig(job_id="eval_" + RUN_NAME, detector="yolov8", detector_weights=str(best),
                segmenter="none",           # boxes are what is scored; skip SAM 2 for speed
                tile=TILE, overlap=0.2, sensor=SENSOR, conf=0.01)   # keep low scores for AP
run_job(job, cfg, verbose=False)

result = evaluate.evaluate(evaluate.load_predictions(job / "output"),
                           evaluate.load_coco_boxes(gold_path), iou_thr=0.5, conf=CONF)
note = (f"{len(gold_coco['images'])} held-out scenes from {DATASET_SOURCE}, sensor={SENSOR}, "
        f"tile={TILE}. Split by scene hash, never seen in training.")
(RUN_DIR / "scene_accuracy_test.json").write_text(json.dumps(result, indent=2))
evaluate.write_accuracy_report(result, RUN_DIR / "ACCURACY_REPORT.md", note)
print(json.dumps(result["overall"], indent=2))
print(open(RUN_DIR / "ACCURACY_REPORT.md").read())

In [ ]:
#@title 7. Publish to the worker: models/best.pt + a model card (only if you accept the numbers)
import hashlib, datetime
PUBLISH = False   #@param {type:"boolean"}
card_out = {"run": RUN_NAME, "created_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
            "base_weights": BASE_WEIGHTS, "weights_sha256_16": hashlib.sha256(best.read_bytes()).hexdigest()[:16],
            "classes": classes, "sensor": SENSOR, "tile": TILE, "overlap": 0.2,
            "dataset": json.loads((RUN_DIR / "dataset_card.json").read_text())["splits"],
            "dataset_source": DATASET_SOURCE, "dataset_licence": DATASET_LICENCE,
            "tile_metrics_test": tile_metrics, "scene_metrics_test": result["overall"],
            "detector_licence": "ultralytics YOLOv8 AGPL-3.0",
            "use_only_on": "imagery like the test set - same sensor, resolution, region type"}
(RUN_DIR / "model_card.json").write_text(json.dumps(card_out, indent=2))
if PUBLISH:
    models = ROOT / "models"; models.mkdir(exist_ok=True)
    if (models / "best.pt").exists():
        shutil.copy(models / "best.pt", models / f"best_before_{RUN_NAME}.pt")   # keep the old one
    shutil.copy(best, models / "best.pt")
    shutil.copy(RUN_DIR / "model_card.json", models / "best_model_card.json")
    print("published. The worker uses it from its next job (its cell 4 picks models/best.pt).")
    print("Set job.json sensor =", SENSOR, "and tile =", TILE, "to match training.")
else:
    print("model card written; tick PUBLISH and re-run to hand this model to the worker")